# Emlak Değerleme Basit Doğrusal Regresyon Modeli

Bu çalışmada emlak değerleme veri seti ayrıntılı biçimde incelenmiş, tüm değişkenlerin dağılımları ve hedef değişkenle ilişkileri görselleştirilmiş, en yüksek korelasyona sahip tek bir bağımsız değişken (alışveriş mesafesi) kullanılarak basit doğrusal regresyon modeli kurulmuş ve model performansı çeşitli grafiklerle değerlendirilmiştir.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

from sklearn.model_selection import train_test_split, cross_val_score, KFold
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

import warnings
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_excel('/content/Real estate valuation data set.xlsx')
df.head()

Bağımsız Değişkenler (Girdi / Özellikler - X):

X1 transaction date: Satış veya işlem tarihi (Örn: 2013.250 = 2013 Mart).

X2 house age: Binanın yaşı (yıl cinsinden).

X3 distance to the nearest MRT station: En yakın metro (MRT) istasyonuna uzaklık (metre cinsinden).

X4 number of convenience stores: Yürüme mesafesindeki bakkal/market sayısı.

X5 latitude / X6 longitude: Konumun coğrafi enlem ve boylam koordinatları.

Bağımlı Değişken (Çıktı / Hedef - Y):

Y house price of unit area: Birim alan başına ev fiyatı.

In [ ]:
df = df.rename(columns={'X1 transaction date': 'tarih', 'X2 house age': 'bina_yası', 'X3 distance to the nearest MRT station': 'ulasım_uzaklık', 'X4 number of convenience stores': 'alısveris_mesafe','X5 latitude': 'enlem', 'X6 longitude': 'boylam', 'Y house price of unit area': 'birim_alan_fiyat'})

In [ ]:
df.columns

In [ ]:
df.info()

In [ ]:
df.duplicated().sum()

In [ ]:
df.describe().T

In [ ]:
print(df.isnull().sum())

In [ ]:
print(df.dtypes)

In [ ]:
df.shape

## Keşifsel Veri Analizi

Bu bölümde her bir değişkenin dağılımı, hedef değişken ile ilişkisi ve değişkenler arasındaki korelasyon yapısı ayrıntılı olarak incelenmiştir.

In [ ]:
sayisal_sutunlar = ['tarih', 'bina_yası', 'ulasım_uzaklık', 'alısveris_mesafe', 'enlem', 'boylam', 'birim_alan_fiyat']

fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for i, sutun in enumerate(sayisal_sutunlar):
    sns.histplot(df[sutun], bins=30, kde=True, ax=axes[i], color='#2980b9')
    axes[i].set_title(f'{sutun} Dağılımı')

for j in range(len(sayisal_sutunlar), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 13))
axes = axes.flatten()

for i, sutun in enumerate(sayisal_sutunlar):
    sns.boxplot(y=df[sutun], ax=axes[i], color='#27ae60')
    axes[i].set_title(f'{sutun} Kutu Grafiği')

for j in range(len(sayisal_sutunlar), len(axes)):
    axes[j].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(10, 8))
corr = df[sayisal_sutunlar].corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm', square=True)
plt.title('Korelasyon Matrisi')
plt.tight_layout()
plt.show()

In [ ]:
target = 'birim_alan_fiyat'
corr_with_target = corr[target].sort_values(ascending=False)
corr_with_target

In [ ]:
plt.figure(figsize=(9, 6))
corr_with_target.drop(target).plot(kind='barh', color='#8e44ad')
plt.xlabel('Hedef Değişkenle Korelasyon')
plt.title('Her Değişkenin Birim Alan Fiyatı ile Korelasyonu')
plt.tight_layout()
plt.show()

In [ ]:
bagimsiz_degiskenler = ['tarih', 'bina_yası', 'ulasım_uzaklık', 'alısveris_mesafe', 'enlem', 'boylam']

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
axes = axes.flatten()

for i, degisken in enumerate(bagimsiz_degiskenler):
    axes[i].scatter(df[degisken], df[target], alpha=0.5, color='#e67e22')
    axes[i].set_xlabel(degisken)
    axes[i].set_ylabel(target)
    axes[i].set_title(f'{degisken} - {target} İlişkisi')

plt.tight_layout()
plt.show()

In [ ]:
plt.subplot(1, 2, 1)
sns.histplot(df[target], bins=10, kde=True)
plt.title(f'{target} Dağılımı')

plt.subplot(1, 2, 2)
sns.boxplot(y=df[target])
plt.title(f'{target} Kutu Grafiği')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(9, 7))
sns.scatterplot(data=df, x='boylam', y='enlem', hue=target, palette='viridis', size=target, sizes=(20, 150))
plt.title('Konuma Göre Birim Alan Fiyatı Dağılımı')
plt.xlabel('Boylam')
plt.ylabel('Enlem')
plt.tight_layout()
plt.show()

## Model Kurulumu

Korelasyon analizine göre hedef değişkenle en güçlü ilişkiye sahip değişken alışveriş mesafesi olduğundan, basit doğrusal regresyon modelinde bu değişken kullanılmıştır.

In [ ]:
X = df[['alısveris_mesafe']].values
y = df['birim_alan_fiyat'].values

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

In [ ]:
print(f'Eğitim seti boyutu: {X_train.shape[0]}')
print(f'Test seti boyutu: {X_test.shape[0]}')

In [ ]:
model_simple = LinearRegression()
model_simple.fit(X_train, y_train)

In [ ]:
y_pred_train = model_simple.predict(X_train)
y_pred_test = model_simple.predict(X_test)

In [ ]:
beta_0 = model_simple.intercept_
beta_1 = model_simple.coef_[0]

In [ ]:
print(f'Kesim Noktası (β₀): {beta_0:.4f}')
print(f'Eğim (β₁): {beta_1:.4f}')
print('Model Denklemi:')
print(f'birim_alan_fiyat = {beta_0:.4f} + {beta_1:.4f} × alısveris_mesafe')

In [ ]:
results = pd.DataFrame({
    'Gerçek fiyat': y_test,
    'Tahmin edilen fiyat': y_pred_test,
    'Fark': y_test - y_pred_test
})
results.head()

In [ ]:
rmse_train = np.sqrt(mean_squared_error(y_train, y_pred_train))
rmse_test = np.sqrt(mean_squared_error(y_test, y_pred_test))
mae_train = mean_absolute_error(y_train, y_pred_train)
mae_test = mean_absolute_error(y_test, y_pred_test)
r2_train = r2_score(y_train, y_pred_train)
r2_test = r2_score(y_test, y_pred_test)

metrikler = pd.DataFrame({
    'Metrik': ['RMSE', 'MAE', 'R²'],
    'Eğitim': [rmse_train, mae_train, r2_train],
    'Test': [rmse_test, mae_test, r2_test]
})
metrikler

## Model Görselleştirmesi ve Artık Analizi

Modelin veriye ne kadar iyi uyduğunu görmek için regresyon doğrusu, artık grafiği, artık dağılımı ve normal olasılık grafiği birlikte incelenmiştir.

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='blue', alpha=0.6, label='Gerçek Veri', s=80)

x_line = np.linspace(X.min(), X.max(), 100).reshape(-1, 1)
y_line = model_simple.predict(x_line)
plt.plot(x_line, y_line, color='red', linewidth=2, label=f'Regresyon Doğrusu: y = {beta_0:.2f} + {beta_1:.2f}x')

plt.xlabel('alısveris_mesafe')
plt.ylabel('birim_alan_fiyat')
plt.title('Basit Doğrusal Regresyon')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
degree_iki_model = make_pipeline(PolynomialFeatures(degree=2), LinearRegression())
degree_iki_model.fit(X_train, y_train)
y_line_iki = degree_iki_model.predict(x_line)

plt.figure(figsize=(10, 6))
plt.scatter(X, y, color='gray', alpha=0.5, label='Gerçek Veri', s=60)
plt.plot(x_line, y_line, color='red', linewidth=2, label='1. Derece (Doğrusal)')
plt.plot(x_line, y_line_iki, color='green', linewidth=2, label='2. Derece (Polinom)')
plt.xlabel('alısveris_mesafe')
plt.ylabel('birim_alan_fiyat')
plt.title('Doğrusal ve 2. Derece Polinom Model Karşılaştırması')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
artiklar = y_test - y_pred_test

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].scatter(y_pred_test, artiklar, alpha=0.6, color='#8e44ad')
axes[0].axhline(y=0, color='red', linestyle='--', linewidth=2)
axes[0].set_xlabel('Tahmin Edilen Fiyat')
axes[0].set_ylabel('Artıklar')
axes[0].set_title('Artık Grafiği')

sns.histplot(artiklar, bins=30, kde=True, ax=axes[1], color='#d35400')
axes[1].set_title('Artıkların Dağılımı')

stats.probplot(artiklar, dist='norm', plot=axes[2])
axes[2].set_title('Normal Olasılık Grafiği (QQ-Plot)')

plt.tight_layout()
plt.show()

## Çapraz Doğrulama

Modelin farklı veri bölünmelerindeki kararlılığını görmek için 5 katlı çapraz doğrulama uygulanmıştır.

In [ ]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
cv_skorlari = cross_val_score(LinearRegression(), X, y, cv=kf, scoring='r2')

plt.figure(figsize=(8, 5))
plt.bar(range(1, 6), cv_skorlari, color='#16a085')
plt.axhline(y=cv_skorlari.mean(), color='red', linestyle='--', label=f'Ortalama R² = {cv_skorlari.mean():.3f}')
plt.xlabel('Kat Numarası')
plt.ylabel('R² Skoru')
plt.title('5 Katlı Çapraz Doğrulama R² Skorları')
plt.xticks(range(1, 6))
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
print(f'Model Test R² Skoru: {r2_test:.4f}')
print(f'Bu model, ev fiyatındaki değişimin yaklaşık %{r2_test*100:.1f}\'ini açıklamaktadır.')

## Sonuç

Tek değişken kullanan bu basit model, birim alan fiyatındaki değişimin bir kısmını açıklayabilmektedir. Artık grafiklerinde belirgin bir örüntü bulunmaması modelin makul bir uyum sağladığını, ancak QQ-plot üzerindeki sapmalar artıkların tam olarak normal dağılmadığını göstermektedir. Daha yüksek doğruluk için tüm bağımsız değişkenlerin kullanıldığı çoklu doğrusal regresyon modeli önerilmektedir.